In [4]:
"""
opt (Gurobi optimization) vs simu (Heuristic simulation) - Fixed version
🔧 Fix: alpha[i,t+1].X → alpha[i,t+1].getAttr('X')
"""

from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 0. Data preparation
# =============================================================================
print("📊 Loading data...")
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')
N = 595
T_years = 7

alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
csem_ids = new_df['csemCur'].head(N).astype(int).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]

Y = 365
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

Total_budget = 1606500
L_total = 43

print(f"✅ Data ready: N={N}, T={T_years}, csemCur={csem_ids}")

# =============================================================================
# 1. opt - Gurobi optimization model (fixed version)
# =============================================================================
print("\nRunning optimization...")
mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 0

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.INTEGER, lb=-1, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# 🔥 Fix: safe attribute access
def get_value(var):
    """Safely get the Gurobi variable value"""
    if hasattr(var, 'X'):
        return var.X
    elif hasattr(var, 'getAttr'):
        return var.getAttr('X')
    return 0.0


# initialization
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i])
    mdl.addConstr(last_action[i,0] == -1)

# alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # no action: natural growth 1.01x
        growth = 1.01 * alpha[i,t-1]
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER maintenance: precise to alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT maintenance: precise to alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t]) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

# Constraints
for t in range(T_years):
    # Budget
    yearly_cost = quicksum(C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t] for i in range(N))
    mdl.addConstr(yearly_cost <= Total_budget, name=f"budget_t{t}")
    
    # Action capacity
    total_actions = quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N))
    mdl.addConstr(total_actions <= L_total, name=f"actions_t{t}")

# Objective function
leak_cost = quicksum((Y * alpha[i,t]) * K[i] * C_leak_t[t] for i in range(N) for t in range(T_years))
action_cost = quicksum(C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t] for i in range(N) for t in range(T_years))
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# Solve
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 3600
mdl.optimize()

# 🔥 opt yearly metrics
opt_metrics = []
if mdl.status == GRB.OPTIMAL:
    opt_total = mdl.objVal
    for t in range(T_years):
        coper_count = sum(get_value(x_COPER[i,t]) > 0.5 for i in range(N))
        mant_count = sum(get_value(x_MANT[i,t]) > 0.5 for i in range(N))
        action_cost_t = sum(C_COPER * get_value(x_COPER[i,t]) + C_MANT * get_value(x_MANT[i,t]) for i in range(N))
        leak_cost_t = sum((Y * get_value(alpha[i,t])) * K[i] * C_leak_t[t] for i in range(N))
        
        opt_metrics.append({
            'year': t+1, 'coper_count': int(coper_count), 'mant_count': int(mant_count),
            'total_actions': int(coper_count + mant_count),
            'action_cost': round(action_cost_t, 0), 'leak_cost': round(leak_cost_t, 0),
            'total_cost': round(action_cost_t + leak_cost_t, 0)
        })
    print(f"✅ opt total cost: {opt_total:,.0f}€")
else:
    print(f"❌ opt optimization failed: {mdl.status}")
    opt_metrics = [{'year': t+1, 'total_cost': 0} for t in range(T_years)]
    opt_total = float('inf')

df_opt = pd.DataFrame(opt_metrics)

# =============================================================================
# 2. simu - Heuristic simulation
# =============================================================================
print("\nRunning simulation...")
simu_metrics = []
csem_results = []

for i in range(N):
    csemCur = csem_ids[i]
    alpha_current = alpha_init[i]
    triggered_previous_year = False
    csem_K = K[i]
    
    for t in range(T_years):
        year = t + 1
        
        # Leak calculation
        leak = alpha_current * Y * csem_K
        delta_p = Delta_P[i]
        interval = 365 / (alpha_current * 365 / delta_p) if alpha_current > 0 else float('inf')
        
        # Trigger condition
        trigger_this_year = (leak >= 5) or (interval <= 60)
        
        # Cost (action_cost is cost of last year's trigger)
        leak_cost = leak * C_leak_t[t]
        action_cost = C_MANT if triggered_previous_year else 0
        total_cost = leak_cost + action_cost
        
        csem_results.append({
            'csem': csemCur, 'year': year, 'total_cost': round(total_cost, 0),
            'action_cost': round(action_cost, 0), 'leak': round(leak, 3),
            'triggered': trigger_this_year
        })
        
        # Update alpha
        if trigger_this_year:
            alpha_current /= mu_MANT
        else:
            alpha_current *= 1.01
        triggered_previous_year = trigger_this_year

# simu yearly aggregation
sim_df = pd.DataFrame(csem_results)
simu_total = sim_df['total_cost'].sum()
for t in range(T_years):
    year_data = sim_df[sim_df['year'] == t+1]
    mant_count = year_data['triggered'].sum()
    action_cost_t = (year_data['action_cost'].sum())
    leak_cost_t = year_data['total_cost'].sum() - action_cost_t
    
    simu_metrics.append({
        'year': t+1, 'coper_count': 0, 'mant_count': int(mant_count),
        'total_actions': int(mant_count),
        'action_cost': round(action_cost_t, 0), 'leak_cost': round(leak_cost_t, 0),
        'total_cost': round(year_data['total_cost'].sum(), 0)
    })

df_simu = pd.DataFrame(simu_metrics)
print(f"✅ simu total cost: {simu_total:,.0f}€")

# =============================================================================
# 3. Combined comparison
# =============================================================================
print("\n" + "="*80)
print(f"📊 Annual comparison results (N={N}, T={T_years})")
print("="*80)

# Merge and compare
comparison = pd.merge(df_opt, df_simu, on='year', suffixes=('_opt', '_simu'))

# Differences
comparison['Cost_diff (opt-simu)'] = (comparison['total_cost_opt']-comparison['total_cost_simu']).round(0)
comparison['Diff%'] = ((comparison['total_cost_opt']-comparison['total_cost_simu']) / 
                         comparison['total_cost_simu'] * 100).round(1)

# Core cost comparison
core_cols = ['year', 'total_cost_opt', 'total_cost_simu', 'Cost_diff (opt-simu)', 'Diff%']
print("\n📈 Annual cost comparison:")
print(comparison[core_cols].round(0))

# Task comparison
task_cols = ['year']
if 'coper_count_opt' in comparison.columns:
    task_cols += ['coper_count_opt', 'coper_count_simu']
if 'mant_count_opt' in comparison.columns:
    task_cols += ['mant_count_opt', 'mant_count_simu']
if 'total_actions_opt' in comparison.columns:
    task_cols += ['total_actions_opt', 'total_actions_simu']

if len(task_cols) > 1:
    print("\n🔧 Annual task comparison:")
    print(comparison[task_cols].round(0))

# Key metrics summary
print("\n🎯 Global key metrics")
print("-" * 70)

metrics_opt = {}
metrics_simu = {}

for col in ['total_cost', 'coper_count', 'mant_count', 'total_actions', 'action_cost', 'leak_cost']:
    metrics_opt[col] = df_opt[col].sum() if col in df_opt.columns else 0
    metrics_simu[col] = df_simu[col].sum() if col in df_simu.columns else 0

save_total = (metrics_opt['total_cost'] - metrics_simu['total_cost']) if metrics_simu['total_cost'] > 0 else 0

print(f"{'Total €':<10} {metrics_opt['total_cost']:>12,.0f} {'vs':<5} {metrics_simu['total_cost']:>12,.0f} {'｜':<5} {save_total:>7.1f}")
print(f"{'Total acts':<10} {int(metrics_opt['total_actions']):>12} {'vs':<5} {int(metrics_simu['total_actions']):>12} {'｜':<5} {int(metrics_opt['total_actions']-metrics_simu['total_actions']):>7}")
print(f"{'COPER':<10} {int(metrics_opt['coper_count']):>12} {'vs':<5} {int(metrics_simu['coper_count']):>12} {'｜':<5} {int(metrics_opt['coper_count']):>7}")
print(f"{'MANT':<10} {int(metrics_opt['mant_count']):>12} {'vs':<5} {int(metrics_simu['mant_count']):>12} {'｜':<5} {int(metrics_opt['mant_count']-metrics_simu['mant_count']):>7}")
print(f"{'Action €':<10} {metrics_opt['action_cost']:>12,.0f} {'vs':<5} {metrics_simu['action_cost']:>12,.0f} {'｜':<5} {int(metrics_opt['action_cost']-metrics_simu['action_cost']):>9}")
print(f"{'Leak €':<10} {metrics_opt['leak_cost']:>12,.0f} {'vs':<5} {metrics_simu['leak_cost']:>12,.0f} {'｜':<5} {int(metrics_opt['leak_cost']-metrics_simu['leak_cost']):>9}")

# print(f"\n🏆 Summary: opt saves {save_total:.1f} of total cost!")

# =============================================================================
# 4. Save results
# =============================================================================
df_opt.to_csv('opt_yearly_metrics.csv', index=False)
df_simu.to_csv('simu_yearly_metrics.csv', index=False)
comparison.to_csv('yearly_comparison.csv', index=False)
sim_df.to_csv('csem_detailed_results.csv', index=False)

print(f"\n✅ 4 result files saved:")
for file in ['opt_yearly_metrics.csv', 'simu_yearly_metrics.csv', 
             'yearly_comparison.csv', 'csem_detailed_results.csv']:
    print(f"   📄 {file}")

print(f"\n🎉 Comparison complete! Optimization model saved {((simu_total-opt_total)/simu_total*100):.1f}% of total cost")


📊 Loading data...
✅ Data ready: N=595, T=7, csemCur=[5000200795, 5000252448, 5000252449, 5000252450, 5000252524, 5000252525, 5000252528, 5000252529, 5000252530, 5000252531, 5000252532, 5000252544, 5000252545, 5000252549, 5000252551, 5000252554, 5000252555, 5000261916, 5000270722, 5000270810, 5000279236, 5000279237, 5000279238, 5000279239, 5000279241, 5000279256, 5000279257, 5000279258, 5000279260, 5000279263, 5000764620, 5000764621, 5000764622, 5000764623, 5000764624, 5000764625, 5000764632, 5000764633, 5000764634, 5000808065, 5000808066, 5000808067, 5000808086, 5000808087, 5000808088, 5000818455, 5000818456, 5000818458, 5000818459, 5000818460, 5000818461, 5000818462, 5000818463, 5000818464, 5000818487, 5000818489, 5000818490, 5000818491, 5000818492, 5000818493, 5000818508, 5000818509, 5000818510, 5000818612, 5000818614, 5000818616, 5000818622, 5000818628, 5000818629, 5000818630, 5000818631, 5000818632, 5000818634, 5000818644, 5000818645, 5000818646, 5000818647, 5000818648, 5000818656,

In [3]:
## new code to compare 4 strategies (opt alpha & cr vs simu alpha & cr) 
# -*- coding: utf-8 -*-
"""
Four-method comparison:
1) opt_alpha
2) opt_cr
3) simu_alpha
4) simu_cr

输出：
- opt_alpha_yearly_metrics.csv
- opt_cr_yearly_metrics.csv
- simu_alpha_yearly_metrics.csv
- simu_cr_yearly_metrics.csv
- four_methods_yearly_comparison.csv
"""

from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
import warnings
warnings.filterwarnings("ignore")


# =============================================================================
# 0. Global settings & data
# =============================================================================
print("📊 Loading data...")

# ===== 你这里改成自己的数据路径 =====
DATA_PATH = "new_df_AlphaCorrected.csv"

new_df = pd.read_csv(DATA_PATH, sep=",")

# ===== 统一对比范围 =====
N = 224
T_years = 17
BASE_YEAR = 2024

# ===== 基础字段 =====
sub_df = new_df.head(N).copy()

alpha_init = sub_df["alpha"].tolist()
V = sub_df["Volume"].tolist()
csem_ids = sub_df["csemCur"].astype(int).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]

Delta_P = [
    round(float(sub_df.iloc[i]["Pression Nominale Relative"] - sub_df.iloc[i]["CR_threshold"]), 3)
    for i in range(N)
]

Age_2023 = sub_df["Age 2023"].tolist()
cr_2023 = sub_df["cr"].tolist()

Y = 365
C_COPER = 15600
C_MANT = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]

mu_COPER = 1.6
mu_MANT = 4.2

Total_budget = 1606500
L_total = 43

gamma_alpha = 5.682011911867158
gamma_beta = 0.25018840842979584

print(f"✅ Data ready: N={N}, T={T_years}")


# =============================================================================
# Utils
# =============================================================================
def compute_gamma_h(age, gamma_alpha=gamma_alpha, gamma_beta=gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z)
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den


def get_gurobi_value(var):
    try:
        return var.X
    except:
        try:
            return var.getAttr("X")
        except:
            return 0.0


def build_metrics_from_detailed(df_detail, year_col="year", task_col="task_type"):
    """
    用于 simulation 输出的详细表 -> 年度汇总
    统一列：
    year, coper_count, mant_count, total_actions, action_cost, leak_cost, total_cost
    """
    metrics = []
    for y in sorted(df_detail[year_col].unique()):
        dfy = df_detail[df_detail[year_col] == y].copy()

        coper_count = (dfy[task_col] == "coper").sum() if task_col in dfy.columns else 0
        mant_count = (dfy[task_col] == "mant").sum() if task_col in dfy.columns else 0
        action_cost_t = dfy["action_cost"].sum() if "action_cost" in dfy.columns else 0
        leak_cost_t = dfy["leak_cost"].sum() if "leak_cost" in dfy.columns else 0
        total_cost_t = dfy["total_cost"].sum() if "total_cost" in dfy.columns else action_cost_t + leak_cost_t

        metrics.append({
            "year": int(y),
            "coper_count": int(coper_count),
            "mant_count": int(mant_count),
            "total_actions": int(coper_count + mant_count),
            "action_cost": round(action_cost_t, 0),
            "leak_cost": round(leak_cost_t, 0),
            "total_cost": round(total_cost_t, 0)
        })

    return pd.DataFrame(metrics)


# =============================================================================
# 1. OPT_ALPHA
# =============================================================================
def run_opt_alpha():
    print("\n🔍 Running opt_alpha...")

    mdl = Model("opt_alpha")
    mdl.Params.OutputFlag = 0
    mdl.Params.MIPGap = 0.05
    mdl.Params.TimeLimit = 300
    mdl.Params.IntFeasTol = 1e-6
    mdl.Params.NumericFocus = 2

    x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
    x_MANT = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
    alpha = mdl.addVars(N, T_years + 1, name="alpha")
    last_action = mdl.addVars(N, T_years + 1, vtype=GRB.INTEGER, lb=-1, ub=T_years, name="last_action")
    cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
    yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

    M_alpha = max(alpha_init) * 10

    # initialization
    for i in range(N):
        h_now = compute_gamma_h(Age_2023[i])
        h_next = compute_gamma_h(Age_2023[i] + 1)

        if h_now > 0:
            alpha_2024 = alpha_init[i] / h_now * h_next
        else:
            alpha_2024 = alpha_init[i]

        mdl.addConstr(alpha[i, 0] == alpha_2024)
        mdl.addConstr(last_action[i, 0] == -1)

    # alpha evolution
    for i in range(N):
        h_now = compute_gamma_h(Age_2023[i])
        for t in range(1, T_years + 1):
            no_action = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])

            if h_now > 0:
                growth = (alpha_init[i] / h_now) * compute_gamma_h(Age_2023[i] + t + 1)
            else:
                growth = alpha_init[i]

            mdl.addConstr(alpha[i, t] >= growth - M_alpha * (1 - no_action))
            mdl.addConstr(alpha[i, t] <= growth + M_alpha * (1 - no_action))

            coper_target = alpha[i, t - 1] / mu_COPER
            mdl.addConstr(alpha[i, t] <= coper_target + M_alpha * (1 - x_COPER[i, t - 1]))
            mdl.addConstr(alpha[i, t] >= coper_target - M_alpha * (1 - x_COPER[i, t - 1]))

            mant_target = alpha[i, t - 1] / mu_MANT
            mdl.addConstr(alpha[i, t] <= mant_target + M_alpha * (1 - x_MANT[i, t - 1]))
            mdl.addConstr(alpha[i, t] >= mant_target - M_alpha * (1 - x_MANT[i, t - 1]))

    # last_action update
    M_time = T_years + 5
    for i in range(N):
        for t in range(1, T_years + 1):
            action_t1 = x_COPER[i, t - 1] + x_MANT[i, t - 1]
            mdl.addConstr(last_action[i, t] >= (t - 1) - M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] <= (t - 1) + M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] >= last_action[i, t - 1] - M_time * action_t1)
            mdl.addConstr(last_action[i, t] <= last_action[i, t - 1] + M_time * action_t1)

    # yearly leak
    BIG_M_LEAK = 1e6
    for i in range(N):
        for t in range(T_years):
            mdl.addConstr(yearly_leak[i, t] == (Y * alpha[i, t] * (10 ** 5)) * K[i])

    # cumulative leak logic
    LEAK_TH = 2.0
    for i in range(N):
        mdl.addConstr(cum_leak[i, 0] == yearly_leak[i, 0])

        for t in range(1, T_years):
            no_action_prev = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])
            mdl.addConstr(
                cum_leak[i, t] >= cum_leak[i, t - 1] + yearly_leak[i, t] - BIG_M_LEAK * (1 - no_action_prev)
            )
            mdl.addConstr(
                cum_leak[i, t] <= cum_leak[i, t - 1] + yearly_leak[i, t] + BIG_M_LEAK * (1 - no_action_prev)
            )

            action_prev = x_COPER[i, t - 1] + x_MANT[i, t - 1]
            mdl.addConstr(cum_leak[i, t] >= yearly_leak[i, t] - BIG_M_LEAK * (1 - action_prev))
            mdl.addConstr(cum_leak[i, t] <= yearly_leak[i, t] + BIG_M_LEAK * (1 - action_prev))

        for t in range(T_years):
            x_this_year = x_COPER[i, t] + x_MANT[i, t]
            mdl.addConstr(x_this_year <= 1)
            mdl.addConstr(cum_leak[i, t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

    # global constraints
    for t in range(T_years):
        yearly_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
        mdl.addConstr(yearly_cost <= Total_budget, name=f"budget_t{t}")

        total_actions = quicksum(x_COPER[i, t] + x_MANT[i, t] for i in range(N))
        mdl.addConstr(total_actions <= L_total, name=f"actions_t{t}")

    # objective
    leak_cost = quicksum(
        (Y * alpha[i, t] * (10 ** 5)) * K[i] * C_leak_t[t]
        for i in range(N) for t in range(T_years)
    )
    action_cost = quicksum(
        C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t]
        for i in range(N) for t in range(T_years)
    )
    mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

    mdl.optimize()

    metrics = []
    detail_rows = []

    if mdl.status == GRB.OPTIMAL or mdl.status == GRB.TIME_LIMIT:
        total_cost = mdl.objVal
        print(f"✅ opt_alpha total cost: {total_cost:,.0f}€")

        for t in range(T_years):
            coper_count = sum(get_gurobi_value(x_COPER[i, t]) > 0.5 for i in range(N))
            mant_count = sum(get_gurobi_value(x_MANT[i, t]) > 0.5 for i in range(N))
            action_cost_t = sum(
                C_COPER * get_gurobi_value(x_COPER[i, t]) + C_MANT * get_gurobi_value(x_MANT[i, t])
                for i in range(N)
            )
            leak_cost_t = sum(
                (Y * get_gurobi_value(alpha[i, t]) * (10 ** 5)) * K[i] * C_leak_t[t]
                for i in range(N)
            )

            metrics.append({
                "year": BASE_YEAR + t,
                "coper_count": int(coper_count),
                "mant_count": int(mant_count),
                "total_actions": int(coper_count + mant_count),
                "action_cost": round(action_cost_t, 0),
                "leak_cost": round(leak_cost_t, 0),
                "total_cost": round(action_cost_t + leak_cost_t, 0)
            })

        for i in range(N):
            for t in range(T_years):
                if get_gurobi_value(x_COPER[i, t]) > 0.5:
                    task_type = "coper"
                    action_cost_i_t = C_COPER
                elif get_gurobi_value(x_MANT[i, t]) > 0.5:
                    task_type = "mant"
                    action_cost_i_t = C_MANT
                else:
                    task_type = None
                    action_cost_i_t = 0

                alpha_val = get_gurobi_value(alpha[i, t])
                leak_val = get_gurobi_value(yearly_leak[i, t])
                leak_cost_i_t = leak_val * C_leak_t[t]

                detail_rows.append({
                    "csemCur": csem_ids[i],
                    "year": BASE_YEAR + t,
                    "alpha": alpha_val,
                    "leak": leak_val,
                    "task_type": task_type,
                    "action_cost": action_cost_i_t,
                    "leak_cost": leak_cost_i_t,
                    "total_cost": action_cost_i_t + leak_cost_i_t
                })

    else:
        print(f"❌ opt_alpha failed: status={mdl.status}")
        total_cost = float("inf")
        metrics = [{"year": BASE_YEAR + t, "total_cost": 0} for t in range(T_years)]

    return pd.DataFrame(metrics), pd.DataFrame(detail_rows), total_cost


# =============================================================================
# 2. OPT_CR
# =============================================================================
def run_opt_cr():
    print("\n🔍 Running opt_cr...")

    mdl = Model("opt_cr")
    mdl.Params.OutputFlag = 0
    mdl.Params.MIPGap = 0.05
    mdl.Params.TimeLimit = 300
    mdl.Params.IntFeasTol = 1e-6
    mdl.Params.NumericFocus = 2

    x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
    x_MANT = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
    last_action = mdl.addVars(N, T_years + 1, vtype=GRB.INTEGER, lb=-1, ub=T_years, name="last_action")
    cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
    yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")
    CR = mdl.addVars(N, T_years + 1, lb=0, name="CR")

    M_cr = 1000

    # initialization
    for i in range(N):
        mdl.addConstr(last_action[i, 0] == -1)
        h_2024 = compute_gamma_h(Age_2023[i] + 1)
        mdl.addConstr(CR[i, 0] == cr_2023[i] + h_2024)

    # CR evolution
    for i in range(N):
        for t in range(1, T_years + 1):
            no_action = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])
            h_cr = compute_gamma_h(Age_2023[i] + t + 1)

            mdl.addConstr(CR[i, t] >= CR[i, t - 1] + h_cr - M_cr * (1 - no_action))
            mdl.addConstr(CR[i, t] <= CR[i, t - 1] + h_cr + M_cr * (1 - no_action))

            coper_target = CR[i, t - 1] / mu_COPER
            mdl.addConstr(CR[i, t] >= coper_target - M_cr * (1 - x_COPER[i, t - 1]))
            mdl.addConstr(CR[i, t] <= coper_target + M_cr * (1 - x_COPER[i, t - 1]))

            mant_target = CR[i, t - 1] / mu_MANT
            mdl.addConstr(CR[i, t] >= mant_target - M_cr * (1 - x_MANT[i, t - 1]))
            mdl.addConstr(CR[i, t] <= mant_target + M_cr * (1 - x_MANT[i, t - 1]))

    # last_action update
    M_time = T_years + 5
    for i in range(N):
        for t in range(1, T_years + 1):
            action_t1 = x_COPER[i, t - 1] + x_MANT[i, t - 1]
            mdl.addConstr(last_action[i, t] >= (t - 1) - M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] <= (t - 1) + M_time * (1 - action_t1))
            mdl.addConstr(last_action[i, t] >= last_action[i, t - 1] - M_time * action_t1)
            mdl.addConstr(last_action[i, t] <= last_action[i, t - 1] + M_time * action_t1)

    # yearly leak
    BIG_M_LEAK = 1e6
    for i in range(N):
        for t in range(T_years):
            mdl.addConstr(yearly_leak[i, t] == (CR[i, t] * Delta_P[i] * (10 ** 5)) * K[i])

    # cumulative leak
    LEAK_TH = 2.0
    for i in range(N):
        mdl.addConstr(cum_leak[i, 0] == yearly_leak[i, 0])

        for t in range(1, T_years):
            no_action_prev = 1 - (x_COPER[i, t - 1] + x_MANT[i, t - 1])
            mdl.addConstr(
                cum_leak[i, t] >= cum_leak[i, t - 1] + yearly_leak[i, t] - BIG_M_LEAK * (1 - no_action_prev)
            )
            mdl.addConstr(
                cum_leak[i, t] <= cum_leak[i, t - 1] + yearly_leak[i, t] + BIG_M_LEAK * (1 - no_action_prev)
            )

            action_prev = x_COPER[i, t - 1] + x_MANT[i, t - 1]
            mdl.addConstr(cum_leak[i, t] >= yearly_leak[i, t] - BIG_M_LEAK * (1 - action_prev))
            mdl.addConstr(cum_leak[i, t] <= yearly_leak[i, t] + BIG_M_LEAK * (1 - action_prev))

        for t in range(T_years):
            x_this_year = x_COPER[i, t] + x_MANT[i, t]
            mdl.addConstr(x_this_year <= 1)
            mdl.addConstr(cum_leak[i, t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

    # global constraints
    for t in range(T_years):
        yearly_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
        mdl.addConstr(yearly_cost <= Total_budget, name=f"budget_t{t}")

        total_actions = quicksum(x_COPER[i, t] + x_MANT[i, t] for i in range(N))
        mdl.addConstr(total_actions <= L_total, name=f"actions_t{t}")

    # objective
    leak_cost = quicksum(
        (CR[i, t] * Delta_P[i] * (10 ** 5)) * K[i] * C_leak_t[t]
        for i in range(N) for t in range(T_years)
    )
    action_cost = quicksum(
        C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t]
        for i in range(N) for t in range(T_years)
    )
    mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

    mdl.optimize()

    metrics = []
    detail_rows = []

    if mdl.status == GRB.OPTIMAL or mdl.status == GRB.TIME_LIMIT:
        total_cost = mdl.objVal
        print(f"✅ opt_cr total cost: {total_cost:,.0f}€")

        for t in range(T_years):
            coper_count = sum(get_gurobi_value(x_COPER[i, t]) > 0.5 for i in range(N))
            mant_count = sum(get_gurobi_value(x_MANT[i, t]) > 0.5 for i in range(N))
            action_cost_t = sum(
                C_COPER * get_gurobi_value(x_COPER[i, t]) + C_MANT * get_gurobi_value(x_MANT[i, t])
                for i in range(N)
            )
            leak_cost_t = sum(
                (get_gurobi_value(CR[i, t]) * Delta_P[i] * (10 ** 5)) * K[i] * C_leak_t[t]
                for i in range(N)
            )

            metrics.append({
                "year": BASE_YEAR + t,
                "coper_count": int(coper_count),
                "mant_count": int(mant_count),
                "total_actions": int(coper_count + mant_count),
                "action_cost": round(action_cost_t, 0),
                "leak_cost": round(leak_cost_t, 0),
                "total_cost": round(action_cost_t + leak_cost_t, 0)
            })

        for i in range(N):
            for t in range(T_years):
                if get_gurobi_value(x_COPER[i, t]) > 0.5:
                    task_type = "coper"
                    action_cost_i_t = C_COPER
                elif get_gurobi_value(x_MANT[i, t]) > 0.5:
                    task_type = "mant"
                    action_cost_i_t = C_MANT
                else:
                    task_type = None
                    action_cost_i_t = 0

                cr_val = get_gurobi_value(CR[i, t])
                leak_val = get_gurobi_value(yearly_leak[i, t])
                leak_cost_i_t = leak_val * C_leak_t[t]

                detail_rows.append({
                    "csemCur": csem_ids[i],
                    "year": BASE_YEAR + t,
                    "cr": cr_val,
                    "leak": leak_val,
                    "task_type": task_type,
                    "action_cost": action_cost_i_t,
                    "leak_cost": leak_cost_i_t,
                    "total_cost": action_cost_i_t + leak_cost_i_t
                })

    else:
        print(f"❌ opt_cr failed: status={mdl.status}")
        total_cost = float("inf")
        metrics = [{"year": BASE_YEAR + t, "total_cost": 0} for t in range(T_years)]

    return pd.DataFrame(metrics), pd.DataFrame(detail_rows), total_cost


# =============================================================================
# 3. SIMU_ALPHA
# =============================================================================
def run_simu_alpha():
    print("\n🎲 Running simu_alpha...")

    new_df_sim = new_df.copy()
    new_df_sim["csemCur"] = new_df_sim["csemCur"].astype(int)

    df_alpha_2023 = new_df_sim.groupby("csemCur")[[
        "alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "Age 2023"
    ]].last().reset_index()

    df_alpha_2023.columns = [
        "csemCur", "alpha_2023", "Pression Nominale Relative",
        "CR_threshold", "Volume", "Age 2023"
    ]

    # 只保留前N个csem
    df_alpha_2023 = df_alpha_2023.head(N).copy()

    nb_coper = 3
    nb_mant = 14

    csem_states = {}
    for _, row in df_alpha_2023.iterrows():
        csemCur = int(row["csemCur"])
        age = row["Age 2023"]

        h_now = compute_gamma_h(age)
        h_next = compute_gamma_h(age + 1)

        if h_now > 0:
            alpha_current = (row["alpha_2023"] / h_now) * h_next
        else:
            alpha_current = row["alpha_2023"]

        csem_states[csemCur] = {
            "alpha_current": alpha_current,
            "pression_nominale": row["Pression Nominale Relative"],
            "cr_threshold": row["CR_threshold"],
            "volume_csem": row["Volume"],
            "csem_K": 146 * row["Volume"] / (8.314 * 293.15 * 10**3),
            "pending_task": None
        }

    rows = []

    for t in range(T_years):
        year = BASE_YEAR + t

        # apply previous pending task
        applied_tasks = {}
        for csemCur, state in csem_states.items():
            if state["pending_task"]:
                task = state["pending_task"]
                if task == "mant":
                    state["alpha_current"] /= mu_MANT
                elif task == "coper":
                    state["alpha_current"] /= mu_COPER
                applied_tasks[csemCur] = task
                state["pending_task"] = None

        # find triggered
        triggered_csem = []
        for csemCur, state in csem_states.items():
            leak = state["alpha_current"] * 365 * (10**5) * state["csem_K"]
            delta_p = state["pression_nominale"] - state["cr_threshold"]
            interval = 365 / (state["alpha_current"] * 365 / delta_p) if state["alpha_current"] > 0 else np.inf

            trigger_this_year = (leak >= 5) or (interval <= 60)
            if trigger_this_year:
                triggered_csem.append({
                    "csemCur": csemCur,
                    "leak": leak,
                    "interval": interval,
                    "state": state
                })

        triggered_csem.sort(key=lambda x: x["leak"], reverse=True)

        task_assignments = {}
        for idx, csem_info in enumerate(triggered_csem):
            csemCur = csem_info["csemCur"]
            if idx < nb_mant:
                task_assignments[csemCur] = "mant"
            elif idx < nb_mant + nb_coper:
                task_assignments[csemCur] = "coper"

        for csemCur in task_assignments:
            csem_states[csemCur]["pending_task"] = task_assignments[csemCur]

        for csemCur, state in csem_states.items():
            leak = state["alpha_current"] * 365 * (10**5) * state["csem_K"]
            delta_p = state["pression_nominale"] - state["cr_threshold"]
            interval = 365 / (state["alpha_current"] * 365 / delta_p) if state["alpha_current"] > 0 else np.inf
            trigger_this_year = (leak >= 5) or (interval <= 60)

            leak_cost = leak * C_leak_t[t]
            task_type = applied_tasks.get(csemCur)
            action_cost = 0
            if task_type == "mant":
                action_cost = C_MANT
            elif task_type == "coper":
                action_cost = C_COPER

            total_cost = leak_cost + action_cost

            rows.append({
                "csemCur": csemCur,
                "year": year,
                "alpha": state["alpha_current"],
                "leak": leak,
                "interval": interval,
                "triggered": trigger_this_year,
                "task_type": task_type,
                "leak_cost": leak_cost,
                "action_cost": action_cost,
                "total_cost": total_cost
            })

    sim_df = pd.DataFrame(rows)
    yearly_metrics = build_metrics_from_detailed(sim_df, year_col="year", task_col="task_type")
    simu_total = sim_df["total_cost"].sum()

    print(f"✅ simu_alpha total cost: {simu_total:,.0f}€")
    return yearly_metrics, sim_df, simu_total


# =============================================================================
# 4. SIMU_CR
# =============================================================================
def run_simu_cr():
    print("\n🎲 Running simu_cr...")

    new_df_sim = new_df.copy()
    new_df_sim["csemCur"] = new_df_sim["csemCur"].astype(int)

    age_map = new_df_sim.groupby("csemCur")["Age 2023"].first().to_dict()

    df_cr_2023 = new_df_sim.groupby("csemCur")[[
        "alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"
    ]].last().reset_index()

    df_cr_2023.columns = [
        "csemCur", "alpha_2023", "Pression Nominale Relative",
        "CR_threshold", "Volume", "cr"
    ]

    df_cr_2023 = df_cr_2023.head(N).copy()

    nb_coper = 3
    nb_mant = 14

    csem_states = {}
    for _, row in df_cr_2023.iterrows():
        csemCur = int(row["csemCur"])
        csem_states[csemCur] = {
            "cr_current": row["cr"] if row["cr"] > 0 else 0,
            "age_2023_fixed": age_map.get(csemCur, 0),
            "current_age": age_map.get(csemCur, 0) + 1,
            "alpha": row["alpha_2023"],
            "pression_nominale": row["Pression Nominale Relative"],
            "cr_threshold": row["CR_threshold"],
            "volume_csem": row["Volume"],
            "csem_K": 146 * row["Volume"] / (8.314 * 293.15 * 10**3)
        }

    rows = []

    for t in range(T_years):
        year = BASE_YEAR + t
        year_offset = year - 2023

        triggered_csem = []
        for csemCur, state in csem_states.items():
            cr = state["cr_current"]
            delta_p = state["pression_nominale"] - state["cr_threshold"]
            leak = cr * delta_p * (10**5) * state["csem_K"]
            interval = 365 / cr if cr > 0 else np.inf

            if leak >= 5 or interval <= 60:
                triggered_csem.append({
                    "csemCur": csemCur,
                    "leak": leak,
                    "state": state
                })

        triggered_csem.sort(key=lambda x: x["leak"], reverse=True)

        task_assignments = {}
        for idx, info in enumerate(triggered_csem):
            csemCur = info["csemCur"]
            if idx < nb_mant:
                task_assignments[csemCur] = "mant"
            elif idx < nb_mant + nb_coper:
                task_assignments[csemCur] = "coper"

        for csemCur, state in csem_states.items():
            cr = state["cr_current"]
            delta_p = state["pression_nominale"] - state["cr_threshold"]
            leak = cr * delta_p * (10**5) * state["csem_K"]
            interval = 365 / cr if cr > 0 else np.inf
            triggered = (leak >= 5) or (interval <= 60)

            task_type = task_assignments.get(csemCur)
            action_cost = {"mant": C_MANT, "coper": C_COPER}.get(task_type, 0)
            leak_cost = leak * C_leak_t[t]
            total_cost = leak_cost + action_cost

            rows.append({
                "csemCur": csemCur,
                "year": year,
                "cr": cr,
                "leak": leak,
                "interval": interval,
                "triggered": triggered,
                "task_type": task_type,
                "leak_cost": leak_cost,
                "action_cost": action_cost,
                "total_cost": total_cost
            })

            # update next year
            next_offset = year_offset + 1
            next_age = state["age_2023_fixed"] + next_offset

            if task_type == "mant":
                state["cr_current"] /= mu_MANT
            elif task_type == "coper":
                state["cr_current"] /= mu_COPER
            else:
                state["cr_current"] += compute_gamma_h(next_age)

            state["current_age"] = next_age

    sim_df = pd.DataFrame(rows)
    yearly_metrics = build_metrics_from_detailed(sim_df, year_col="year", task_col="task_type")
    simu_total = sim_df["total_cost"].sum()

    print(f"✅ simu_cr total cost: {simu_total:,.0f}€")
    return yearly_metrics, sim_df, simu_total


# =============================================================================
# 5. Run all four methods
# =============================================================================
df_opt_alpha, detail_opt_alpha, total_opt_alpha = run_opt_alpha()
df_opt_cr, detail_opt_cr, total_opt_cr = run_opt_cr()
df_simu_alpha, detail_simu_alpha, total_simu_alpha = run_simu_alpha()
df_simu_cr, detail_simu_cr, total_simu_cr = run_simu_cr()


# =============================================================================
# 6. Combined comparison
# =============================================================================
print("\n" + "=" * 100)
print(f"📊 Four-method annual comparison (N={N}, T={T_years})")
print("=" * 100)

comparison = (
    df_opt_alpha.rename(columns={
        "coper_count": "coper_count_opt_alpha",
        "mant_count": "mant_count_opt_alpha",
        "total_actions": "total_actions_opt_alpha",
        "action_cost": "action_cost_opt_alpha",
        "leak_cost": "leak_cost_opt_alpha",
        "total_cost": "total_cost_opt_alpha"
    })
    .merge(
        df_opt_cr.rename(columns={
            "coper_count": "coper_count_opt_cr",
            "mant_count": "mant_count_opt_cr",
            "total_actions": "total_actions_opt_cr",
            "action_cost": "action_cost_opt_cr",
            "leak_cost": "leak_cost_opt_cr",
            "total_cost": "total_cost_opt_cr"
        }),
        on="year", how="outer"
    )
    .merge(
        df_simu_alpha.rename(columns={
            "coper_count": "coper_count_simu_alpha",
            "mant_count": "mant_count_simu_alpha",
            "total_actions": "total_actions_simu_alpha",
            "action_cost": "action_cost_simu_alpha",
            "leak_cost": "leak_cost_simu_alpha",
            "total_cost": "total_cost_simu_alpha"
        }),
        on="year", how="outer"
    )
    .merge(
        df_simu_cr.rename(columns={
            "coper_count": "coper_count_simu_cr",
            "mant_count": "mant_count_simu_cr",
            "total_actions": "total_actions_simu_cr",
            "action_cost": "action_cost_simu_cr",
            "leak_cost": "leak_cost_simu_cr",
            "total_cost": "total_cost_simu_cr"
        }),
        on="year", how="outer"
    )
    .sort_values("year")
    .reset_index(drop=True)
)

cost_cols = [
    "year",
    "total_cost_opt_alpha",
    "total_cost_opt_cr",
    "total_cost_simu_alpha",
    "total_cost_simu_cr"
]

print("\n📈 Annual total cost comparison:")
print(comparison[cost_cols].round(0))

print("\n🔧 Annual action comparison:")
action_cols = [
    "year",
    "total_actions_opt_alpha",
    "total_actions_opt_cr",
    "total_actions_simu_alpha",
    "total_actions_simu_cr"
]
print(comparison[action_cols].fillna(0).round(0))


# =============================================================================
# 7. Global summary
# =============================================================================
print("\n🎯 Global summary")
print("-" * 100)

summary_df = pd.DataFrame([
    {
        "method": "opt_alpha",
        "total_cost": df_opt_alpha["total_cost"].sum(),
        "action_cost": df_opt_alpha["action_cost"].sum(),
        "leak_cost": df_opt_alpha["leak_cost"].sum(),
        "total_actions": df_opt_alpha["total_actions"].sum(),
        "coper_count": df_opt_alpha["coper_count"].sum(),
        "mant_count": df_opt_alpha["mant_count"].sum()
    },
    {
        "method": "opt_cr",
        "total_cost": df_opt_cr["total_cost"].sum(),
        "action_cost": df_opt_cr["action_cost"].sum(),
        "leak_cost": df_opt_cr["leak_cost"].sum(),
        "total_actions": df_opt_cr["total_actions"].sum(),
        "coper_count": df_opt_cr["coper_count"].sum(),
        "mant_count": df_opt_cr["mant_count"].sum()
    },
    {
        "method": "simu_alpha",
        "total_cost": df_simu_alpha["total_cost"].sum(),
        "action_cost": df_simu_alpha["action_cost"].sum(),
        "leak_cost": df_simu_alpha["leak_cost"].sum(),
        "total_actions": df_simu_alpha["total_actions"].sum(),
        "coper_count": df_simu_alpha["coper_count"].sum(),
        "mant_count": df_simu_alpha["mant_count"].sum()
    },
    {
        "method": "simu_cr",
        "total_cost": df_simu_cr["total_cost"].sum(),
        "action_cost": df_simu_cr["action_cost"].sum(),
        "leak_cost": df_simu_cr["leak_cost"].sum(),
        "total_actions": df_simu_cr["total_actions"].sum(),
        "coper_count": df_simu_cr["coper_count"].sum(),
        "mant_count": df_simu_cr["mant_count"].sum()
    }
])

print(summary_df.round(0))

best_method = summary_df.sort_values("total_cost").iloc[0]
print(f"\n🏆 Best method: {best_method['method']} | total cost = {best_method['total_cost']:,.0f}€")


# =============================================================================
# 8. Save results
# =============================================================================
# df_opt_alpha.to_csv("opt_alpha_yearly_metrics.csv", index=False)
# df_opt_cr.to_csv("opt_cr_yearly_metrics.csv", index=False)
# df_simu_alpha.to_csv("simu_alpha_yearly_metrics.csv", index=False)
# df_simu_cr.to_csv("simu_cr_yearly_metrics.csv", index=False)

# detail_opt_alpha.to_csv("opt_alpha_detailed_results.csv", index=False)
# detail_opt_cr.to_csv("opt_cr_detailed_results.csv", index=False)
# detail_simu_alpha.to_csv("simu_alpha_detailed_results.csv", index=False)
# detail_simu_cr.to_csv("simu_cr_detailed_results.csv", index=False)

# comparison.to_csv("four_methods_yearly_comparison.csv", index=False)
# summary_df.to_csv("four_methods_global_summary.csv", index=False)

print("\n✅ Saved files:")
for file in [
    "opt_alpha_yearly_metrics.csv",
    "opt_cr_yearly_metrics.csv",
    "simu_alpha_yearly_metrics.csv",
    "simu_cr_yearly_metrics.csv",
    "opt_alpha_detailed_results.csv",
    "opt_cr_detailed_results.csv",
    "simu_alpha_detailed_results.csv",
    "simu_cr_detailed_results.csv",
    "four_methods_yearly_comparison.csv",
    "four_methods_global_summary.csv"
]:
    print(f"   📄 {file}")

print("\n🎉 Four-method comparison complete!")

📊 Loading data...
✅ Data ready: N=224, T=17

🔍 Running opt_alpha...
✅ opt_alpha total cost: 19,453,731€

🔍 Running opt_cr...

Interrupt request received
❌ opt_cr failed: status=11

🎲 Running simu_alpha...
✅ simu_alpha total cost: 12,944,911€

🎲 Running simu_cr...
✅ simu_cr total cost: 41,525,890€

📊 Four-method annual comparison (N=224, T=17)

📈 Annual total cost comparison:
    year  total_cost_opt_alpha  total_cost_opt_cr  total_cost_simu_alpha  \
0   2024              968881.0                  0               875281.0   
1   2025              925424.0                  0              1298941.0   
2   2026             1004225.0                  0               792219.0   
3   2027              951646.0                  0               573623.0   
4   2028              850748.0                  0               595027.0   
5   2029             1129612.0                  0               616431.0   
6   2030             1041256.0                  0               637834.0   
7   2031      

KeyError: "['total_actions_opt_cr'] not in index"